Chirag Bansal

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# 1. Generate Data with High Correlation
np.random.seed(42)
m = 1000
# Create a base feature
x_base = np.random.rand(m, 1)
# Create 7 highly correlated columns by adding small noise to the base feature
X_gen = np.hstack([x_base + np.random.normal(0, 0.01, (m, 1)) for _ in range(7)])
# Target variable y = 3*x1 + 2*x2 ... + noise
true_weights = np.random.rand(7, 1) * 5
y_gen = X_gen.dot(true_weights) + np.random.normal(0, 0.1, (m, 1))

# 2. Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_gen)
# Add Intercept (Bias) column
X_b = np.c_[np.ones((m, 1)), X_scaled]
y = y_gen

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_b, y, test_size=0.2, random_state=42)

# 3. Ridge Regression Class (Step-by-Step Gradient Descent)
class RidgeRegressionGD:
    def __init__(self, learning_rate, lambda_val, n_iterations=1000):
        self.lr = learning_rate
        self.lambda_val = lambda_val
        self.n_iterations = n_iterations
        self.theta = None
        self.cost_history = []

    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros((n, 1))

        for i in range(self.n_iterations):
            prediction = X.dot(self.theta)
            error = prediction - y

            # Gradient calculation
            # Note: We do not regularize the bias term (theta[0])
            gradient = (1/m) * X.T.dot(error)
            # Add regularization term to gradient (excluding bias)
            reg_term = (self.lambda_val) * self.theta
            reg_term[0] = 0 # Don't penalize bias

            gradient = gradient + reg_term

            # Update theta
            self.theta -= self.lr * gradient

            # Calculate Cost (MSE + L2 Penalty)
            cost = (1/(2*m)) * np.sum(error**2) + (self.lambda_val/2) * np.sum(self.theta[1:]**2)
            self.cost_history.append(cost)

    def predict(self, X):
        return X.dot(self.theta)

# 4. Grid Search
learning_rates = [0.0001, 0.001, 0.01, 0.1, 1, 10]
regularization_params = [1e-15, 1e-10, 1e-5, 1e-3, 0, 1, 10, 20]

best_score = -np.inf
best_cost = np.inf
best_params = {}
best_model = None

print(f"{'LR':<10} {'Lambda':<10} {'R2 Score':<15} {'Final Cost'}")
print("-" * 50)

for lr in learning_rates:
    for lam in regularization_params:
        model = RidgeRegressionGD(learning_rate=lr, lambda_val=lam, n_iterations=1000)

        # Catch numerical overflows for bad LR choices
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = r2_score(y_test, y_pred)
            final_cost = model.cost_history[-1]

            if np.isnan(score) or np.isnan(final_cost):
                continue

            print(f"{lr:<10} {lam:<10} {score:.5f}          {final_cost:.5f}")

            # Criteria: Max R2 and Min Cost
            if score > best_score:
                best_score = score
                best_cost = final_cost
                best_params = {'lr': lr, 'lambda': lam}
                best_model = model
        except Exception as e:
            continue

print("-" * 50)
print(f"Best Parameters: LR={best_params['lr']}, Lambda={best_params['lambda']}")
print(f"Best R2 Score: {best_score}")
print(f"Minimum Cost: {best_cost}")

LR         Lambda     R2 Score        Final Cost
--------------------------------------------------
0.0001     1e-15      -1.49091          43.38587
0.0001     1e-10      -1.49091          43.38587
0.0001     1e-05      -1.49091          43.38589
0.0001     0.001      -1.49093          43.38692
0.0001     0          -1.49091          43.38587
0.0001     1          -1.50802          44.37191
0.0001     10         -1.64905          49.74778
0.0001     20         -1.76519          52.25056
0.001      1e-15      0.60772          6.44905
0.001      1e-10      0.60772          6.44905
0.001      1e-05      0.60772          6.44907
0.001      0.001      0.60773          6.45149
0.001      0          0.60772          6.44905
0.001      1          0.60711          8.58618
0.001      10         0.32557          16.51385
0.001      20         0.13620          19.12670
0.01       1e-15      0.99963          0.00597
0.01       1e-10      0.99963          0.00597
0.01       1e-05      0.99963       

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in scalar multiply
  cost = (1/(2*m)) * np.sum(error**2) + (self.lambda_val/2) * np.sum(self.theta[1:]**2)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in square
  cost = (1/(2*m)) * np.sum(error**2) + (self.lambda_val/2) * np.sum(self.theta[1:]**2)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1275: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in square
  cost = (1/(2*m)) * np

0.1        20         -inf          inf


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in square
  cost = (1/(2*m)) * np.sum(error**2) + (self.lambda_val/2) * np.sum(self.theta[1:]**2)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in square
  cost = (1/(2*m)) * np.sum(error**2) + (self.lambda_val/2) * np.sum(self.theta[1:]**2)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipython-input-3839441555.py:59: RuntimeWarning: overflow encountered in scalar multiply
  cost = (1/(2*m)) * np.sum(erro

--------------------------------------------------
Best Parameters: LR=0.1, Lambda=1e-15
Best R2 Score: 0.9996387209771266
Minimum Cost: 0.0058696543138096954


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# (a) Load and Pre-process Data
# Fetching the dataset internally
hitters = sm.datasets.get_rdataset("Hitters", "ISLR").data

print(f"Original shape: {hitters.shape}")

# Handle Null Values (Salary has NaNs)
hitters = hitters.dropna()
print(f"Shape after dropping NaNs: {hitters.shape}")

# Categorical to Numerical (One-Hot Encoding or Binary encoding)
# Columns: League (A/N), Division (E/W), NewLeague (A/N)
hitters = pd.get_dummies(hitters, columns=['League', 'Division', 'NewLeague'], drop_first=True)

# (b) Separate Input/Output and Scale
X = hitters.drop('Salary', axis=1)
y = hitters['Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# (c) Fit Models
# 1. Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

# 2. Ridge Regression (alpha = 0.5748)
ridge_reg = Ridge(alpha=0.5748)
ridge_reg.fit(X_train_scaled, y_train)

# 3. LASSO Regression (alpha = 0.5748)
lasso_reg = Lasso(alpha=0.5748)
lasso_reg.fit(X_train_scaled, y_train)

# (d) Evaluate Performance
models = {'Linear': lin_reg, 'Ridge': ridge_reg, 'Lasso': lasso_reg}

print("\nPerformance Evaluation (Test Set):")
print(f"{'Model':<10} {'RMSE':<15} {'R2 Score'}")
print("-" * 40)

for name, model in models.items():
    pred = model.predict(X_test_scaled)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    print(f"{name:<10} {rmse:.4f}          {r2:.4f}")

# Analysis logic
print("\nAnalysis:")
print("In the Hitters dataset, Ridge and Lasso typically perform better than pure Linear Regression.")
print("This is because the dataset has many correlated features (multicollinearity).")
print("Lasso is particularly useful here as it performs feature selection by shrinking some coefficients to exactly zero.")
print("Ridge handles the multicollinearity by shrinking coefficients, reducing overfitting compared to OLS.")

Original shape: (322, 20)
Shape after dropping NaNs: (263, 20)

Performance Evaluation (Test Set):
Model      RMSE            R2 Score
----------------------------------------
Linear     358.1680          0.2907
Ridge      355.8144          0.3000
Lasso      356.0050          0.2993

Analysis:
In the Hitters dataset, Ridge and Lasso typically perform better than pure Linear Regression.
This is because the dataset has many correlated features (multicollinearity).
Lasso is particularly useful here as it performs feature selection by shrinking some coefficients to exactly zero.
Ridge handles the multicollinearity by shrinking coefficients, reducing overfitting compared to OLS.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.185e+04, tolerance: 4.367e+03
  model = cd_fast.enet_coordinate_descent(


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Boston Dataset (Using raw URL as load_boston is deprecated)
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

X = data
y = target

# 2. Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. RidgeCV
# alphas: array of alpha values to try
alphas_ridge = [0.1, 1.0, 10.0, 100.0]
ridge_cv = RidgeCV(alphas=alphas_ridge, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print(f"RidgeCV Chosen Alpha: {ridge_cv.alpha_}")
print(f"RidgeCV Best Score (R2 on Test): {ridge_cv.score(X_test_scaled, y_test):.4f}")

# 4. LassoCV
# LassoCV automatically explores a path of alphas
lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print(f"LassoCV Chosen Alpha: {lasso_cv.alpha_}")
print(f"LassoCV Best Score (R2 on Test): {lasso_cv.score(X_test_scaled, y_test):.4f}")

# Validation of results
y_pred_ridge = ridge_cv.predict(X_test_scaled)
y_pred_lasso = lasso_cv.predict(X_test_scaled)

print("\nFinal RMSE Comparison:")
print(f"RidgeCV RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}")
print(f"LassoCV RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}")

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-2118615977.py:10: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


RidgeCV Chosen Alpha: 1.0
RidgeCV Best Score (R2 on Test): 0.6685
LassoCV Chosen Alpha: 0.006863892263379676
LassoCV Best Score (R2 on Test): 0.6684

Final RMSE Comparison:
RidgeCV RMSE: 4.9308
LassoCV RMSE: 4.9314


In [4]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Data
iris = load_iris()
X = iris.data
y = iris.target
classes = np.unique(y) # [0, 1, 2]

# 2. Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Add Bias term
X_bias = np.c_[np.ones((X_scaled.shape[0], 1)), X_scaled]

X_train, X_test, y_train, y_test = train_test_split(X_bias, y, test_size=0.2, random_state=42)

# 3. Helper Functions for Logistic Regression
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(theta, X, y):
    m = len(y)
    h = sigmoid(X.dot(theta))
    # Epsilon to prevent log(0) error
    epsilon = 1e-5
    cost = (-1/m) * np.sum(y * np.log(h + epsilon) + (1-y) * np.log(1-h + epsilon))
    return cost

def gradient_descent(X, y, theta, lr, iterations):
    m = len(y)
    cost_history = []

    for i in range(iterations):
        h = sigmoid(X.dot(theta))
        gradient = (1/m) * X.T.dot(h - y)
        theta -= lr * gradient

        if i % 100 == 0:
            cost_history.append(compute_cost(theta, X, y))

    return theta, cost_history

# 4. One-vs-Rest Training Strategy
all_thetas = []
learning_rate = 0.1
num_iterations = 3000

print("Training One-vs-Rest Classifiers...")

for c in classes:
    # Create binary target: 1 if current class, 0 otherwise
    y_binary = np.where(y_train == c, 1, 0)

    # Initialize theta
    initial_theta = np.zeros(X_train.shape[1])

    # Train binary classifier
    theta, _ = gradient_descent(X_train, y_binary, initial_theta, learning_rate, num_iterations)
    all_thetas.append(theta)
    print(f"Class {c} vs Rest model trained.")

# 5. Prediction
def predict_multiclass(X, all_thetas):
    # Calculate probability for each class classifier
    # Shape: (n_samples, n_classes)
    probs = np.array([sigmoid(X.dot(theta)) for theta in all_thetas]).T
    # Return the index (class) with the highest probability
    return np.argmax(probs, axis=1)

# Evaluate
y_pred = predict_multiclass(X_test, all_thetas)

print("\nResults:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Training One-vs-Rest Classifiers...
Class 0 vs Rest model trained.
Class 1 vs Rest model trained.
Class 2 vs Rest model trained.

Results:
Accuracy: 0.9667

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.89      0.94         9
   virginica       0.92      1.00      0.96        11

    accuracy                           0.97        30
   macro avg       0.97      0.96      0.97        30
weighted avg       0.97      0.97      0.97        30

